# Titanic Survival Prediction — Baseline

Notebook bằng tiếng **Tiếng Việt**.

**Mục tiêu:**
- Chuẩn bị dữ liệu & xử lý đặc trưng
- Huấn luyện mô hình baseline: Logistic Regression

Chạy tất cả các ô (Run All) sau khi đặt file `train.csv` (và tùy chọn `test.csv`) vào thư mục làm việc.

In [65]:
# Thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import joblib

# hiển thị đồ thị inline
%matplotlib inline
print("Ready")

Ready


In [66]:
# Load dữ liệu (place train.csv vào cùng thư mục notebook)
import os
print("Working dir:", os.getcwd())

try:
    train = pd.read_csv("train.csv")
    display(train.head())
    print("\nMissing summary:")
    print(train.isnull().sum())
except FileNotFoundError:
    print("Không tìm thấy file train.csv. Vui lòng đặt file train.csv vào thư mục làm việc và chạy lại notebook.")

Working dir: /content
Không tìm thấy file train.csv. Vui lòng đặt file train.csv vào thư mục làm việc và chạy lại notebook.


## 1. Khảo sát nhanh (EDA)
Quan sát các biến, phân phối và mối quan hệ với `Survived`.

In [67]:
# Nếu train đã được load, thực hiện EDA cơ bản
if 'train' in globals():
    print('Shape:', train.shape)
    print('\nCác giá trị duy nhất mỗi cột:')
    print(train.nunique())
    # Tỷ lệ sống sót theo giới tính và hạng
    display(pd.crosstab(train['Sex'], train['Survived'], normalize='index'))
    display(pd.crosstab(train['Pclass'], train['Survived'], normalize='index'))
    # Biểu đồ tuổi
    train['Age'].hist(bins=30)
    plt.title('Age distribution')
    plt.xlabel('Age')
    plt.show()

## 2. Xử lý đặc trưng (Feature Engineering)
- Điền missing
- Tạo biến mới: `FamilySize`, `IsAlone`, `Title` (tách từ `Name`)
- Loại bỏ các cột ít hữu ích trực tiếp như `Ticket`, `Cabin` (có thể dùng sau này)

In [68]:
if 'train' in globals():
    df = train.copy()
    # Tạo FamilySize và IsAlone
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # Extract Title từ Name
    df['Title'] = df['Name'].str.extract(',\s*([^\.]+)\.', expand=False)
    # Gom nhóm các title ít gặp
    df['Title'] = df['Title'].replace(['Mlle','Ms'],'Miss')
    df['Title'] = df['Title'].replace('Mme','Mrs')
    rare_titles = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
    df['Title'] = df['Title'].replace(list(rare_titles), 'Rare')

    display(df[['Name','Title']].head())
    display(df[['FamilySize','IsAlone']].head())

<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-713177670.py:8: SyntaxWarning: invalid escape sequence '\s'
  df['Title'] = df['Name'].str.extract(',\s*([^\.]+)\.', expand=False)


### 3. Chọn features cho baseline
Chúng ta dùng các feature đơn giản: `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`, `FamilySize`, `IsAlone`, `Title`.

In [69]:
if 'train' in globals():
    features = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked','FamilySize','IsAlone','Title']
    X = df[features].copy()
    y = df['Survived'].copy()
    X.head()

### 4. Pipeline tiền xử lý
- Impute numerical bằng median
- Impute categorical bằng mode
- One-hot encode categorical
- Standardize numeric trước khi đưa vào Logistic Regression

In [70]:
from sklearn.pipeline import make_pipeline

if 'train' in globals():
    numeric_features = ['Age','SibSp','Parch','Fare','FamilySize']
    categorical_features = ['Pclass','Sex','Embarked','IsAlone','Title']

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

In [71]:
if 'train' in globals():
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    # Build pipeline with logistic regression
    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('classifier', LogisticRegression(max_iter=1000))])
    clf.fit(X_train, y_train)
    print("Done training baseline Logistic Regression")

In [72]:
if 'train' in globals():
    y_pred = clf.predict(X_val)
    y_proba = clf.predict_proba(X_val)[:,1]
    print("Accuracy:", accuracy_score(y_val, y_pred))
    print("\nClassification report:\n", classification_report(y_val, y_pred))
    print("\nConfusion matrix:\n", confusion_matrix(y_val, y_pred))
    print("\nROC AUC:", roc_auc_score(y_val, y_proba))

    # Plot ROC curve
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0,1],[0,1],'--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC curve')
    plt.show()

### 5. Cross-validation (tùy chọn)
Để kiểm tra ổn định, ta có thể chạy cross-val trên pipeline.

In [73]:
if 'train' in globals():
    scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
    print("CV accuracy scores:", scores)
    print("Mean CV accuracy:", scores.mean())

### 6. Lưu model
Lưu pipeline đã huấn luyện để dùng lại.

In [74]:
if 'train' in globals():
    joblib.dump(clf, 'titanic_logreg_baseline.joblib')
    print("Saved model to titanic_logreg_baseline.joblib")